<a href="https://colab.research.google.com/github/Raymondycp/Scarping_EcomerceDatabase/blob/main/Web_Scraping_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import lxml
import pandas as pd
from datetime import datetime

In [ ]:
#get all product page list
pageres = requests.get('https://www.lungfung.hk/foodndrink/snacks.html')
pagesoup = BeautifulSoup(pageres.text, 'lxml')
pageurllist = []
pageurl_elements = pagesoup.select('.page')
for pageurls in pageurl_elements :
    pageurllist.append(pageurls.get('href'))
pageurllist = [item for item in pageurllist if item is not None]
pageurllist = list(set(pageurllist))
pageurllist.extend(['https://www.lungfung.hk/foodndrink/snacks.html?p=1','https://www.lungfung.hk/foodndrink/snacks.html?p=6','https://www.lungfung.hk/foodndrink/snacks.html?p=7'])
print(pageurllist)

['https://www.lungfung.hk/foodndrink/snacks.html?p=2', 'https://www.lungfung.hk/foodndrink/snacks.html?p=5', 'https://www.lungfung.hk/foodndrink/snacks.html?p=3', 'https://www.lungfung.hk/foodndrink/snacks.html?p=4', 'https://www.lungfung.hk/foodndrink/snacks.html?p=1', 'https://www.lungfung.hk/foodndrink/snacks.html?p=6', 'https://www.lungfung.hk/foodndrink/snacks.html?p=7']


In [ ]:
#get all product list
productslist = []
for productsurl in pageurllist:
    productsres = requests.get(productsurl)
    productssoup = BeautifulSoup(productsres.text, 'lxml')

    for productsurl in productssoup.select('.product-item-link'):
        productslist.append(productsurl.get('href'))

# Now productslist contains all the scraped URLs
# print(productslist)
print(len(productslist))


224


In [ ]:
#scrape all products into a dic
products_data = []  # 用於儲存所有產品的數據
# x = 0

for url in productslist:
    # x += 1
    res2 = requests.get(url)
    soup2 = BeautifulSoup(res2.text, 'lxml')

    dic = {}  # 每個產品的字典

    # Extracting the required data
    dic['Brand'] = soup2.select_one('.brand-title').text.strip() if soup2.select_one('.brand-title') else None  # Brand
    dic['Name'] = soup2.find('div', itemprop='description').text.strip() if soup2.find('div', itemprop='description') else None  # Value
    dic['NameZH'] = soup2.find('h1').text.strip() if soup2.find('h1') else None  # Name
    dic['Origin'] = soup2.find('td', {'data-th': '品牌國家'}).text.strip() if soup2.find('td', {'data-th': '品牌國家'}) else None  # Origin
    dic['Currency'] = soup2.find('meta', itemprop='priceCurrency')['content'] if soup2.find('meta', itemprop='priceCurrency') else None  # PriceCurrency
    dic['Price'] = soup2.find('meta', itemprop='price')['content'] if soup2.find('meta', itemprop='price') else None  # Price
    dic['Stock'] = soup2.find('div', class_='availability only').strong.text.strip() if soup2.find('div', class_='availability only') else None  # Stock
    dic['Discount'] = soup2.select_one('h2').text.strip() if soup2.select_one('h2') else None  # Discount
    dic['Description'] = soup2.select_one('.description').text.strip() if soup2.select_one('.description') else None  # Description
    dic['Sku'] = soup2.find('div', itemprop='sku').text.strip() if soup2.find('div', itemprop='sku') else None # SKU

    # Append the product dictionary to the products_data list
    products_data.append(dic)


In [ ]:
#turn dic to dataframe
raw_df = pd.DataFrame(products_data)
raw_df['Date'] = pd.Timestamp.today().normalize()  # This gives the date without time

raw_df['website_name'] = 'LF'
#Raw df
#Export DataFrame to CSV
df = raw_df.copy()


###  Save rawfile

In [ ]:
#export raw df to csv
today_date = datetime.now().strftime('%Y-%m-%d')  # Format: YYYY-MM-DD
rawfilename = f"LungFung_raw_data_{today_date}.csv"
raw_df.to_csv(rawfilename, index=False)  # index=False

### ReadCSV

In [ ]:
df= pd.read_csv('LungFung_raw_data_2024-10-07.csv')

### Clean CSV

In [ ]:
'''Data Cleaning'''
#add date column into dataframe###################################
df['Size'] = raw_df['NameZH'].str.split().str[-1]
df['Size'] =df['Size'].str.extract(r'(\d+)')#Add size value
df['Unit'] = raw_df['NameZH'].str.extract(r'(\d+)(G)')[1]
df['NameZH'] = raw_df['NameZH'].str.replace(r'\d+G', '', regex=True) #Del g in name
df['Name'] = raw_df['Name'].str.replace(r'\d+G', '', regex=True)
df['BrandZH'] = raw_df['Brand'].str.extract(r'([\u4e00-\u9fa5]+)')
df['Brand'] = raw_df['Brand'].str.replace(r'[\u4e00-\u9fa5]+', '', regex=True)

##################################################################
df = df[['BrandZH','Brand', 'Name', 'NameZH', 'Origin', 'Currency', 'Price', 'Stock','Discount', 'Description', 'Sku', 'Size',
'Unit', 'Date', 'website_name']]

In [ ]:
# import numpy as np
# df=df.fillna(value='None')
# df.info()
# df[df['Brand'].isna()]
# df = df.applymap(lambda x: None if x == '' else x)
# Remove single quotes from all elements
df = df.replace("'", " ", regex=True)

In [ ]:
# df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BrandZH       61 non-null     object        
 1   Brand         224 non-null    object        
 2   Name          81 non-null     object        
 3   NameZH        224 non-null    object        
 4   Origin        79 non-null     object        
 5   Currency      224 non-null    object        
 6   Price         224 non-null    object        
 7   Stock         185 non-null    object        
 8   Discount      224 non-null    object        
 9   Description   81 non-null     object        
 10  Sku           224 non-null    object        
 11  Size          210 non-null    object        
 12  Unit          172 non-null    object        
 13  Date          224 non-null    datetime64[us]
 14  website_name  224 non-null    object        
dtypes: datetime64[us](1), object(14)
memory 

In [ ]:
# 定義關鍵字到類別的映射及其優先級
category_keywords = {
    '薯片': (['蝦條', '薯條', '芋片', '薯片', '蝦片'], 1),
    '脆片類': (['脆片', '粟米片', '蝦片', '脆脆','爆米花','爆谷'], 5),
    '堅果類': (['果仁', '開心果', '花生', "果仁", "腰果", "核桃仁", "核桃", "花生米", "花生", "子仁", "籽仁", '杏', '仁'], 11),
    '布丁': (['布丁', '布甸', '蒟蒻', '啫喱', '奶凍'], 9),
    '餅乾': (['餅', '先貝', '曲奇', '百力滋', '米果', '小饅頭', '威化', '米餅','批'], 2),
    '肉乾': (['腸', '魚', '蟹', '肉', "腸仔", '肉乾', "魷魚絲", "胸肉", "豬肉", "肉紙", "肉粒"], 4),
    '糕點': (['蛋糕', '麵包', '牛角包', '泡芙', '鳳梨酥', '鳳爪', "吐司",'年糕'], 8),
    '紫菜': (['紫菜', '海苔'], 10),
    '蛋類': (['鵪鶉蛋', '鐵蛋', '蛋', "蛋捲", "鵪鶉蛋", "鳳凰卷"], 10),
    '朱古力': (['朱古力', '巧克力'], 6),
    '糖類': (['糖', '糖果', '軟糖', '牛奶糖', '花生糖', '硬糖', '清熱糖', '香口膠'], 3)
}

# 函數來根據關鍵字確定類別
def get_category(NameZH):
    # product_name = product_name.lower()  # 小寫轉換以便不區分大小寫
    found_categories = []

    for category, (keywords, priority) in category_keywords.items():
        for keyword in keywords:
            if keyword in NameZH:
                found_categories.append((category, priority))
                break  # 找到一個關鍵字後不再檢查該類別

    # 根據優先級返回最優先的類別
    if found_categories:
        return min(found_categories, key=lambda x: x[1])[0]  # 返回優先級最低的類別

    return "Other"  # 如果沒有匹配，返回其他


# 應用函數到產品名稱列以創建新的Category列
df['Category']=df['NameZH'].apply(get_category)



In [ ]:
df

,BrandZH,Brand,Name,NameZH,Origin,Currency,Price,Stock,Discount,Description,Sku,Size,Unit,Date,website_name,Category
0,NaN,,None,濱田健康俱樂部迷你薯仔餅乾 - 淡鹽味,None,HKD,12,6,滿$600免運費,None,FDN250078,64,G,2025-04-16,LF,餅乾
1,NaN,,None,濱田健康俱樂部迷你杏仁巧克力,None,HKD,12,6,滿$600免運費,None,FDN250079,64,G,2025-04-16,LF,朱古力
2,NaN,,None,ASAHI 朝日玄米餅 - 乳酪味,None,HKD,14,7,滿$600免運費,None,FDN250082,72,G,2025-04-16,LF,餅乾
3,NaN,,None,百邦 ELISE 威化棒 - KIRI奶油芝士味 32'S,None,HKD,20,6,滿$600免運費,None,FDN250085,32,NaN,2025-04-16,LF,餅乾
4,NaN,,None,百邦白朱古力夾心曲奇,None,HKD,14,9,滿$600免運費,None,FDN250086,NaN,NaN,2025-04-16,LF,餅乾
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,NaN,,None,森永 HI-CHEW 軟糖 - 草莓味 12'S,None,HKD,9,None,滿$600免運費,None,FDN182979,12,NaN,2025-04-16,LF,糖類
220,NaN,,None,固力果三兄弟甜筒朱古力餅 10'S,None,HKD,24,None,滿$600免運費,None,FDN180753,10,NaN,2025-04-16,LF,餅乾
221,NaN,,None,味覺糖彩虹彈珠糖,None,HKD,9,None,滿$600免運費,None,FDN231316,30,G,2025-04-16,LF,糖類
222,NaN,,None,NABISCO 利士夾心餅-芝士,None,HKD,10,None,滿$600免運費,None,FDN231261,51,G,2025-04-16,LF,餅乾


## Export Cleaned CSV

In [ ]:
#export raw df to csv
today_date = datetime.now().strftime('%Y-%m-%d')  # Format: YYYY-MM-DD
rawfilename = f"LungFung_cleaned_data_{today_date}.csv"
df.to_csv(rawfilename, index=False)  # index=False

In [ ]:
df= pd.read_csv('LungFung_cleaned_data_2024-10-09.csv')
# df.info()

df

,BrandZH,Brand,Name,NameZH,Origin,Currency,Price,Stock,Discount,Description,Sku,Size,Unit,Date,website_name,Category
0,NaN,NaN,NaN,旭製菓 ANTHONY S 爆米花-朱古力杏仁味,NaN,HKD,22,5.0,滿$600免運費,NaN,FDN220916,45.0,G,2024-10-07,LF,脆片類
1,NaN,NaN,NaN,ORIHIRO 零卡路里蒟蒻-西柚味,NaN,HKD,9,4.0,滿$600免運費,NaN,FDN230223,130.0,G,2024-10-07,LF,布丁
2,NaN,NaN,NaN,日清 CISCO 椰子餅乾 16 S,NaN,HKD,9,7.0,滿$600免運費,NaN,FDN222424,16.0,NaN,2024-10-07,LF,餅乾
3,NaN,NaN,NaN,REBISCO HANSEL 特級夾心餅-芝士,NaN,HKD,13,12.0,滿$600免運費,NaN,FDN191136,127.0,G,2024-10-07,LF,餅乾
4,宏源,HONG YUAN,SHENPI TANG,陳皮糖,中國,HKD,23,11.0,滿$600免運費,口感獨特，酸甜可口,FDN190577,355.0,G,2024-10-07,LF,糖類
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,三進,SAMJIN,CHOCO & PEANUT PIE 6PCS,朱古力年糕批-花生醬夾心 6PCS,韓國,HKD,20,NaN,滿$600免運費,韓國直送嘅Samjin 花生醬夾心年糕批做到外脆內軟，加上軟心餡＋超級煙韌嘅年糕，完全抵抗唔...,FDN201877,6.0,NaN,2024-10-07,LF,餅乾
213,杰克,JACKER,POTATO CRISP (ORIGINAL),杰克洋芋片-原味,馬來西亞,HKD,15,NaN,滿$600免運費,商品特色\r\n獨特原味\r\n濃郁美味的香脆爽勁\r\n不油不膩，吮指回味\r\n馬來西亞...,FDN190364,160.0,G,2024-10-07,LF,薯片
214,維多利,GEL,FRUIT PLUS SWEET,水果超軟糖,香港,HKD,19,NaN,滿$600免運費,糖果溶化後仍然留在口中,FDN180271,500.0,G,2024-10-07,LF,糖類
215,丸玉,MARUTAMA,CRAB FISH SNACK,北海道長腳蟹柳,日本,HKD,9,NaN,滿$600免運費,日本進口丸玉蟹肉棒，即食熟食海鮮，製作原料是北海道長腳蟹及海魚，自然鮮味，口感細膩。日式風味...,FDN180701,45.0,G,2024-10-07,LF,肉乾


In [ ]:
# df.columns
# Create a new DataFrame with mapped values
master_df = pd.DataFrame({
    'brand_name': df['BrandZH'],
    'brand_name_en': df['Brand'],
    'brand_description': None,  # Assuming this is used for brand description
    'product_category': df['Category'],  # Placeholder; fill in as needed
    'product_name': df['NameZH'],
    'product_name_en': df['Name'],
    'product_description': df['Description'],  # Assuming this is used for brand description
    'unit_size': df['Size'],        # Placeholder; fill in as needed
    'unit': df['Unit'],
    'country_origin': df['Origin'],
    'availability': None,      # Placeholder; fill in as needed
    'currency': df['Currency'] ,  # Calculating promotion price
    'promotion_price': df['Price'] ,  # Calculating promotion price
    'promotion':df['Discount'],
    'unit_price': df['Price'],
    'sold_qty': None,         # Placeholder; fill in as needed
    'number_of_stock': df['Stock'],
    'product_id': df['Sku'],
    'website_name': df['website_name'],
    'collect_date': df['Date']
})

# Display the master DataFrame

# Export Master CSV

In [ ]:
master_df.fillna(value='None',inplace=True)

In [ ]:
master_df

,brand_name,brand_name_en,brand_description,product_category,product_name,product_name_en,product_description,unit_size,unit,country_origin,availability,currency,promotion_price,promotion,unit_price,sold_qty,number_of_stock,product_id,website_name,collect_date
0,NaN,,None,餅乾,濱田健康俱樂部迷你薯仔餅乾 - 淡鹽味,None,None,64,G,None,None,HKD,12,滿$600免運費,12,None,6,FDN250078,LF,2025-04-16
1,NaN,,None,朱古力,濱田健康俱樂部迷你杏仁巧克力,None,None,64,G,None,None,HKD,12,滿$600免運費,12,None,6,FDN250079,LF,2025-04-16
2,NaN,,None,餅乾,ASAHI 朝日玄米餅 - 乳酪味,None,None,72,G,None,None,HKD,14,滿$600免運費,14,None,7,FDN250082,LF,2025-04-16
3,NaN,,None,餅乾,百邦 ELISE 威化棒 - KIRI奶油芝士味 32'S,None,None,32,NaN,None,None,HKD,20,滿$600免運費,20,None,6,FDN250085,LF,2025-04-16
4,NaN,,None,餅乾,百邦白朱古力夾心曲奇,None,None,NaN,NaN,None,None,HKD,14,滿$600免運費,14,None,9,FDN250086,LF,2025-04-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,NaN,,None,糖類,森永 HI-CHEW 軟糖 - 草莓味 12'S,None,None,12,NaN,None,None,HKD,9,滿$600免運費,9,None,None,FDN182979,LF,2025-04-16
220,NaN,,None,餅乾,固力果三兄弟甜筒朱古力餅 10'S,None,None,10,NaN,None,None,HKD,24,滿$600免運費,24,None,None,FDN180753,LF,2025-04-16
221,NaN,,None,糖類,味覺糖彩虹彈珠糖,None,None,30,G,None,None,HKD,9,滿$600免運費,9,None,None,FDN231316,LF,2025-04-16
222,NaN,,None,餅乾,NABISCO 利士夾心餅-芝士,None,None,51,G,None,None,HKD,10,滿$600免運費,10,None,None,FDN231261,LF,2025-04-16


In [ ]:
#export raw df to csv
today_date = datetime.now().strftime('%Y-%m-%d')  # Format: YYYY-MM-DD
rawfilename = f"LungFung_cleaned_data_master_{today_date}.csv"
master_df.to_csv(rawfilename, index=False)  # index=False
master_df

,brand_name,brand_name_en,brand_description,product_category,product_name,product_name_en,product_description,unit_size,unit,country_origin,availability,currency,promotion_price,promotion,unit_price,sold_qty,number_of_stock,sku,website_name,Date
0,None,None,None,脆片類,旭製菓 ANTHONY S 爆米花-朱古力杏仁味,None,None,45.0,G,None,None,HKD,22,滿$600免運費,22,None,5.0,FDN220916,LF,2024-10-07
1,None,None,None,布丁,ORIHIRO 零卡路里蒟蒻-西柚味,None,None,130.0,G,None,None,HKD,9,滿$600免運費,9,None,4.0,FDN230223,LF,2024-10-07
2,None,None,None,餅乾,日清 CISCO 椰子餅乾 16 S,None,None,16.0,None,None,None,HKD,9,滿$600免運費,9,None,7.0,FDN222424,LF,2024-10-07
3,None,None,None,餅乾,REBISCO HANSEL 特級夾心餅-芝士,None,None,127.0,G,None,None,HKD,13,滿$600免運費,13,None,12.0,FDN191136,LF,2024-10-07
4,宏源,HONG YUAN,None,糖類,陳皮糖,SHENPI TANG,口感獨特，酸甜可口,355.0,G,中國,None,HKD,23,滿$600免運費,23,None,11.0,FDN190577,LF,2024-10-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,三進,SAMJIN,None,餅乾,朱古力年糕批-花生醬夾心 6PCS,CHOCO & PEANUT PIE 6PCS,韓國直送嘅Samjin 花生醬夾心年糕批做到外脆內軟，加上軟心餡＋超級煙韌嘅年糕，完全抵抗唔...,6.0,None,韓國,None,HKD,20,滿$600免運費,20,None,None,FDN201877,LF,2024-10-07
213,杰克,JACKER,None,薯片,杰克洋芋片-原味,POTATO CRISP (ORIGINAL),商品特色\r\n獨特原味\r\n濃郁美味的香脆爽勁\r\n不油不膩，吮指回味\r\n馬來西亞...,160.0,G,馬來西亞,None,HKD,15,滿$600免運費,15,None,None,FDN190364,LF,2024-10-07
214,維多利,GEL,None,糖類,水果超軟糖,FRUIT PLUS SWEET,糖果溶化後仍然留在口中,500.0,G,香港,None,HKD,19,滿$600免運費,19,None,None,FDN180271,LF,2024-10-07
215,丸玉,MARUTAMA,None,肉乾,北海道長腳蟹柳,CRAB FISH SNACK,日本進口丸玉蟹肉棒，即食熟食海鮮，製作原料是北海道長腳蟹及海魚，自然鮮味，口感細膩。日式風味...,45.0,G,日本,None,HKD,9,滿$600免運費,9,None,None,FDN180701,LF,2024-10-07


In [ ]:
df = pd.read_csv('LungFung_cleaned_data_master_2024-10-09.csv')
df.fillna(value='None',inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   brand_name           217 non-null    object
 1   brand_name_en        217 non-null    object
 2   brand_description    217 non-null    object
 3   product_category     217 non-null    object
 4   product_name         217 non-null    object
 5   product_name_en      217 non-null    object
 6   product_description  217 non-null    object
 7   unit_size            217 non-null    object
 8   unit                 217 non-null    object
 9   country_origin       217 non-null    object
 10  availability         217 non-null    object
 11  currency             217 non-null    object
 12  promotion_price      217 non-null    int64 
 13  promotion            217 non-null    object
 14  unit_price           217 non-null    int64 
 15  sold_qty             217 non-null    object
 16  number_o

C:\Users\genhk\AppData\Local\Temp\ipykernel_37864\3509542775.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'None' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna(value='None',inplace=True)


In [ ]:
import gdown
# Specify the file ID extracted from the Google Drive URL
file_id = '1MbAXGM25COQpi1l6jotaDDLIvMxmiZE8'
# Construct the direct download URL
url = f'https://drive.google.com/uc?id={file_id}'
# Download the file
output = 'netflix_titles.csv'  # The name of the downloaded file
gdown.download(url, output, quiet=False)

## Df to Database

In [ ]:
from sqlalchemy import create_engine

In [ ]:
username = 'postgres'
password = ''
database = 'midterim'
host = 'localhost'  # or your host IP
port = '5432'       # default port for PostgreSQL

# Create the connection string
conn_string = f'postgresql://{username}:{password}@{host}:{port}/{database}'

In [ ]:
engine = create_engine(conn_string)


In [ ]:
df1 = pd.read_csv('LungFung_cleaned_data_master_2024-10-09.csv')
df2 = pd.read_csv('Wellcome2024-10-09.csv')
df3 = pd.read_csv('PNS_chips_2024-10-07.csv')
df4 = pd.read_csv('HKTV_MALL32501.csv')


In [ ]:
df5 = pd.read_csv('Sliver_All_.csv')
df5

,brand_name,product_category,product_name,product_description,unit_size,country_origin,availability,promotion_price,promotion_text,unit_price,...,product_id,website_name,collect_date,product_id.1,rating,review_number,promotion_text.1,brand_name.1,collect_date.1,promotion_text.2
0,NaN,朱古力,旭製菓 ANTHONY'S 爆米花-朱古力杏仁味,NaN,45.0,NaN,NaN,22,滿$600免運費,22,...,FDN220916,LF,2024-10-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,布丁,ORIHIRO 零卡路里蒟蒻-西柚味,NaN,130.0,NaN,NaN,9,滿$600免運費,9,...,FDN230223,LF,2024-10-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,餅乾,日清 CISCO 椰子餅乾 16'S,NaN,16.0,NaN,NaN,9,滿$600免運費,9,...,FDN222424,LF,2024-10-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,餅乾,REBISCO HANSEL 特級夾心餅-芝士,NaN,127.0,NaN,NaN,13,滿$600免運費,13,...,FDN191136,LF,2024-10-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,宏源,糖類,陳皮糖,口感獨特，酸甜可口,355.0,中國,NaN,23,滿$600免運費,23,...,FDN190577,LF,2024-10-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1519,EAT EAST,Other,EAT EAST 紅豆沙 (冷凍 0-4°C),NaN,NaN,NaN,True,20.0,NaN,25.9,...,NaN,PNS,NaN,NaN,5.0,NaN,NaN,NaN,2024-10-09,NaN
1520,EAT EAST,Other,EAT EAST 芝麻糊 (冷凍 0-4°C),NaN,NaN,NaN,True,20.0,NaN,25.9,...,NaN,PNS,NaN,NaN,0.0,NaN,NaN,NaN,2024-10-09,NaN
1521,達樂美,布丁,達樂美 濃味芒果果凍啫喱,NaN,NaN,NaN,True,8.4,NaN,8.4,...,NaN,PNS,NaN,NaN,5.5,NaN,NaN,NaN,2024-10-09,NaN
1522,康力施洛,布丁,康力施洛 葡萄及橙味萬聖節版蒟蒻啫哩,NaN,NaN,NaN,True,17.9,NaN,24.9,...,NaN,PNS,NaN,NaN,0.0,NaN,NaN,NaN,2024-10-09,NaN


In [ ]:
df1.columns

Index(['brand_name', 'brand_name_en', 'brand_description', 'product_category',
       'product_name', 'product_name_en', 'product_description', 'unit_size',
       'unit', 'country_origin', 'availability', 'currency', 'promotion_price',
       'promotion', 'unit_price', 'sold_qty', 'number_of_stock', 'product_id',
       'website_name', 'collect_date'],
      dtype='object')

In [ ]:
# df1.rename(columns={'sku': 'product_id'}, inplace=True)
# df1.rename(columns={'Date': 'collect_date'}, inplace=True)

In [ ]:
# df4['collect_date'] = '2024-10-07'
df4.rename(columns={'brand': 'brand_name'}, inplace=True)

In [ ]:
df1.astype(str).to_sql('lf2', engine, if_exists='append', index=False)
# df[['product_name']].to_sql('product_details', engine, if_exists='append', index=False)

217

In [ ]:
df2.astype(str).to_sql('welcome', engine, if_exists='replace', index=False)

954

##Concate


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# df5 = pd.read_csv('Silver_Data.csv')
# df5
df4

,Unnamed: 0,product_id,product_name,unit_price,promotion_price,unit_size,sold_qty,rating,review_number,store_name,...,availability,discount_text,gift_text,buyMoreSaveMore,ParterPromotion,country_origin,product_category,brand_name,website_name,collect_date
0,0,V95770,馬百良 - 馬百良草本清熱糖 42克,"原價$18,今日89折",16.0,42克,3000,4.7,17,馬百良葯廠,...,True,NaN,"1PM前落單,即晚收貨",NaN,NaN,台灣,糖類,馬百良,hktvmall,2024-10-07
1,1,X75811,好侍 - 通加利粟米筒(粟米味) X 2盒 (綠),$ 48.00,29.5,NaN,10000,4.7,54,熊印屋,...,True,NaN,"1PM前落單,即晚收貨",NaN,+$40換2支泰國純椰青水1L,日本,Other,好侍,hktvmall,2024-10-07
2,2,W14250,喜瑪拉雅 - 糖 · e閃購 超冰涼薄荷 (蜂蜜青檬 15gX12) 蜂蜜青檬 天然鹽糖 B...,$ 79.80,60.0,15g x 12,7000,4.5,62,eSoular.com,...,True,NaN,"1PM前落單,即晚收貨",NaN,NaN,馬來西亞,糖類,喜瑪拉雅,hktvmall,2024-10-07
3,3,AE29670,小牧味屋 - Ⓕ · 醇香原味酒鬼花生 (藍18克20袋) 真空 獨立包裝 ~4895090...,搜尋e閃購 筍貨日日有,58.0,18g x 20bags,3000,4.5,44,eSoular.com,...,True,NaN,"1PM前落單,即晚收貨",NaN,$9.9起換購小肥羊清湯湯底,中國,堅果類,小牧味屋,hktvmall,2024-10-07
4,4,Z44552,DENROKU - 日本北海之味什錦米果(12小包) 到期日2024-11-03,限時大劈價~,40.0,NaN,30000,4.5,292,NaN,...,True,NaN,"1PM前落單,即晚收貨",NaN,+$40換2支泰國純椰青水1L,日本,餅乾,DENROKU,hktvmall,2024-10-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32496,32496,BG46464,Binggrae - 肉乾味脆片 70g (Exp.2024.11.09) [平行進口] (EU),$ 25.00,11.0,NaN,0,0.0,0,雜貨大王,...,False,NaN,NaN,NaN,NaN,韓國,脆片類,Binggrae,hktvmall,2024-10-07
32497,32497,BG46467,Binggrae - 煙燻煙肉脆片 70g (Exp.2024.11.14) [平行進口] ...,$ 25.00,11.0,NaN,0,0.0,0,飯氣攻屋,...,False,NaN,NaN,NaN,NaN,韓國,脆片類,Binggrae,hktvmall,2024-10-07
32498,32498,BG46471,Binggrae - 煙燻煙肉脆片 70g (Exp.2024.11.14) [平行進口] ...,$ 25.00,11.0,NaN,0,0.0,0,雜貨大王,...,False,NaN,NaN,NaN,NaN,韓國,脆片類,Binggrae,hktvmall,2024-10-07
32499,32499,AK23391,Oreo - Thins 薄脆朱古力夾心曲奇 - 提拉米蘇味 (2件/盒) [韓版] 獨立包裝,$ 26.00,22.0,42g x 2,50,1.0,1,九哥士多,...,False,NaN,NaN,NaN,NaN,韓國,朱古力,Oreo,hktvmall,2024-10-07


In [ ]:
result_vertical = pd.concat([df1, df2, df3, df4], ignore_index=True)


In [ ]:
df_master_join = result_vertical.copy()

In [ ]:
df_master_join.columns

Index(['brand_name', 'brand_name_en', 'brand_description', 'product_category',
       'product_name', 'product_name_en', 'product_description', 'unit_size',
       'unit', 'country_origin', 'availability', 'currency', 'promotion_price',
       'promotion', 'unit_price', 'sold_qty', 'number_of_stock', 'product_id',
       'website_name', 'collect_date', 'discount', 'storage', 'origin',
       'delivery_method_1', 'delivery_method_2', 'product_brand', 'url',
       'avaliability', 'is_discounted', 'rating', 'review_count', 'hash_tags',
       'Unnamed: 0', 'review_number', 'store_name', 'delivery_time',
       'discount_text', 'gift_text', 'buyMoreSaveMore', 'ParterPromotion'],
      dtype='object')

In [ ]:
df_master_join.product_category.value_counts()

product_category
Other       16042
糖類           3496
餅乾           3249
薯片           2931
堅果類          1623
朱古力          1619
紫菜           1111
脆片類          1002
布丁            977
肉乾            917
糕點            550
薯片/膨脹食品類      235
蛋類            224
果仁              5
乾類              2
Name: count, dtype: int64

In [ ]:
# df_master_join['brand_name'].value_counts()
# # df_master_join['product_category'].value_counts
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from matplotlib import font_manager



In [ ]:
sns.countplot(data=df_master_join, x='product_category')

# Set the title and labels
plt.title('Count of Products by Category')
plt.xlabel('Product Category')
plt.ylabel('Count')

# Show the plot
plt.show()

ModuleNotFoundError: No module named 'pandas.core.methods.to_dict'

In [ ]:
df_master_join['product_category'] = df_master_join['product_category'].astype(str)
# Create a count plot for product categories
plt.figure(figsize=(36, 10))  # Set figure size
sns.countplot(x='product_category', data=df_master_join)  # Correctly pass DataFrame
plt.title('Count of Products by Category')  # Add a title
plt.xticks(rotation=90)  # Rotate x labels for better readability
plt.xlabel('Product Category')  # Label x-axis
plt.ylabel('Count')  # Label y-axis
plt.show()  # Display the plot

ModuleNotFoundError: No module named 'pandas.core.methods.to_dict'

<Figure size 3600x1000 with 0 Axes>

In [ ]:
df_master_join.to_csv('Silver', index=False)  # index=False


In [ ]:
df_master_join.drop(columns=['B', 'C']

In [ ]:
import os
import pandas as pd
from pandasai import Agent
from pandasai import SmartDataframe

ModuleNotFoundError: No module named 'yaml'

In [ ]:
df_master_join

# By default, unless you choose a different LLM, it will use BambooLLM.
# You can get your free API key signing up at https://pandabi.ai (you can also configure it in your .env file)
os.environ["PANDASAI_API_KEY"] = "$2a$10$jNzdLNXDkvcPFlTKKEPJ3eAJKt/W/Xrn4pUIMFVLGTPRVOC1VtMRO"

agent = Agent(sales_by_country)
agent.chat('Which are the top 5 countries by sales?')

In [ ]:
import pandas as pd
from postgre import connect, insertDF
from psycopg2 import Error
import psycopg2.extras as extras


database = "midterim"
host = "localhost"
user = "postgres"
pwd = ""
conn_obj = connect(host, database, user, pwd)


'''
THis is an example of how to insert data into a database table

THis is a table script from SILVER team

    product_description INT, <---this is probably a typo, it should be VARCHAR
	product_id INT PRIMARY KEY,
    category_id INT,
    brand_id INT,
    product_name VARCHAR(255) NOT NULL,
    unit_size INT,
    country_origin VARCHAR(255),
    FOREIGN KEY (category_id) REFERENCES Product_Category(category_id),
    FOREIGN KEY (brand_id) REFERENCES Brand_Dimension(brand_id)
'''

connecting to localhost - midterm
You are connecting to ('PostgreSQL 16.4, compiled by Visual C++ build 1940, 64-bit',)


'\nTHis is an example of how to insert data into a database table\n\nTHis is a table script from SILVER team\n\n    product_description INT, <---this is probably a typo, it should be VARCHAR\n\tproduct_id INT PRIMARY KEY,\n    category_id INT,\n    brand_id INT,\n    product_name VARCHAR(255) NOT NULL,\n    unit_size INT,\n    country_origin VARCHAR(255),\n    FOREIGN KEY (category_id) REFERENCES Product_Category(category_id),\n    FOREIGN KEY (brand_id) REFERENCES Brand_Dimension(brand_id)\n'